# 02 · Gráficos y slides gerenciales
Genera métricas, gráficos y un PPT ejecutivo de 1–2 slides para acompañar el reporte comercial. Parte de los PDFs de Mantra en `MyDrive/mine_chatbot` y guarda outputs en `_analysis_outputs`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/ai_assistant
!git clone -q https://github.com/dinatalediego/ai_assistant.git /content/ai_assistant
%cd /content/ai_assistant
!pip -q install pandas pymupdf matplotlib python-pptx pyarrow pyyaml


In [ ]:
from pathlib import Path
import re, pandas as pd, fitz, matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE

ROOT=Path('/content/drive/MyDrive/mine_chatbot')
OUT=ROOT/'_analysis_outputs'; OUT.mkdir(exist_ok=True)
FIG=OUT/'figures'; FIG.mkdir(exist_ok=True)
inventory=pd.read_csv(OUT/'file_inventory.csv')
pdfs=inventory[inventory['suffix'].str.lower().eq('.pdf')].copy()
print('PDFs detectados:', len(pdfs))

def parse_filename(name):
    stem=Path(name).stem; parts=stem.split('_')
    seq=int(parts[0]) if parts and parts[0].isdigit() else None
    phone=parts[-1] if parts and parts[-1].isdigit() else None
    lead=' '.join(parts[1:-1]) if len(parts)>2 else None
    return seq, lead, phone

def pdf_text(path):
    try:
        doc=fitz.open(path); text='\n'.join(p.get_text('text') for p in doc)
        return text, len(doc), None
    except Exception as exc:
        return '', 0, f'{type(exc).__name__}: {exc}'

records=[]
for _,row in pdfs.iterrows():
    text,pages,error=pdf_text(row['path']); seq,lead,phone=parse_filename(row['name'])
    records.append({'source_file':row['name'],'source_path':row['path'],'file_sequence_candidate':seq,'lead_name_candidate':lead,'lead_numeric_candidate_hashable':phone,'pages':pages,'text_chars':len(text),'text_extractable':len(text.strip())>100,'text':text,'error':error})
pdf_text_df=pd.DataFrame(records)
pdf_text_df.to_parquet(OUT/'pdf_text.parquet', index=False)
pdf_text_df.drop(columns=['text']).to_csv(OUT/'pdf_extractability_profile.csv', index=False)
print('Extractables:', int(pdf_text_df.text_extractable.sum()), 'de', len(pdf_text_df))

def infer(row):
    t=str(row['text']); lead=str(row.get('lead_name_candidate') or '')
    m=re.search(r'Mensajes:\s*(\d+)',t); msgs=int(m.group(1)) if m else None
    projects=re.findall(r'Brochure-([A-Za-zÁÉÍÓÚáéíóúñÑ]+)\.pdf',t)
    advisors=re.findall(r'Te saluda ([^,\n]+?)(?:,| tu asesor|\s+tu asesor)',t)
    replied=bool(re.search(r'\n[A-Z]\n'+re.escape(lead)+r'\n(?!\+51)(.+?)\n\d{2}/\d{2}/\d{4}',t,re.S)) if lead else False
    return pd.Series({'messages_declared':msgs,'project_last':projects[-1] if projects else 'NC','project_count':len(set(projects)),'advisor_last':advisors[-1] if advisors else 'NC','lead_replied':replied,'need_observed':bool(re.search(r'\b[123]\s*dormitorio[s]?\b|presupuesto|precio|cuota|financiamiento',t,re.I)),'visit_or_handoff_offered':bool(re.search(r'agendar|coordinar|visita|cita|sala de ventas|en sala',t,re.I)),'technical_issue_signal':bool(re.search(r'inconveniente técnico|no puedo mostrar|fallido',t,re.I)),'failed_delivery_signal':bool(re.search(r'fallido',t,re.I))})
metrics=pd.concat([pdf_text_df.drop(columns=['text']), pdf_text_df.apply(infer,axis=1)],axis=1)
metrics.to_csv(OUT/'conversation_metrics_v01.csv', index=False)
display(metrics.head())

base=metrics[metrics.text_extractable].copy(); N=len(base)
summary=pd.DataFrame({'etapa':['PDFs legibles','Lead respondió','Conversación >=3 mensajes','Necesidad observable','Invitación/visita'],'n':[N,int(base.lead_replied.sum()),int((base.messages_declared.fillna(0)>=3).sum()),int(base.need_observed.sum()),int(base.visit_or_handoff_offered.sum())]})
summary['pct']=summary.n/max(N,1); summary.to_csv(OUT/'management_funnel.csv',index=False); display(summary)

plt.figure(figsize=(8,3.5)); plt.barh(summary.etapa[::-1], summary.n[::-1])
for i,v in enumerate(summary.n[::-1]): plt.text(v+0.2,i,f'{v} ({v/max(N,1):.0%})',va='center')
plt.title('Funnel conversacional — Mantra'); plt.tight_layout(); plt.savefig(FIG/'conversation_funnel.png',dpi=200); plt.show()

proj=base.project_last.value_counts().sort_values(); plt.figure(figsize=(7,3.5)); plt.barh(proj.index, proj.values)
for i,v in enumerate(proj.values): plt.text(v+0.2,i,str(v),va='center')
plt.title('Conversaciones por proyecto consultado'); plt.tight_layout(); plt.savefig(FIG/'conversation_projects.png',dpi=200); plt.show()

depth=pd.cut(base.messages_declared.fillna(0),[0,1,3,6,999],labels=['1 mensaje','2–3 mensajes','4–6 mensajes','7+ mensajes']).value_counts().sort_index()
plt.figure(figsize=(6,3.5)); plt.bar(depth.index.astype(str),depth.values)
for i,v in enumerate(depth.values): plt.text(i,v+0.2,str(v),ha='center')
plt.title('Profundidad de conversación'); plt.tight_layout(); plt.savefig(FIG/'conversation_depth.png',dpi=200); plt.show()

prs=Presentation(); prs.slide_width=Inches(13.333); prs.slide_height=Inches(7.5)
NAVY=RGBColor(0,48,68); ORANGE=RGBColor(244,126,32); GREY=RGBColor(96,96,96)
def title(slide,t,sub=''):
    tb=slide.shapes.add_textbox(Inches(.55),Inches(.35),Inches(9.5),Inches(.55)); r=tb.text_frame.paragraphs[0].add_run(); r.text=t; r.font.size=Pt(24); r.font.bold=True; r.font.color.rgb=NAVY
    st=slide.shapes.add_textbox(Inches(.57),Inches(.88),Inches(11.5),Inches(.35)); r=st.text_frame.paragraphs[0].add_run(); r.text=sub; r.font.size=Pt(11); r.font.color.rgb=GREY
    mk=slide.shapes.add_shape(MSO_SHAPE.ISOSCELES_TRIANGLE,Inches(12.2),Inches(.35),Inches(.45),Inches(.45)); mk.fill.solid(); mk.fill.fore_color.rgb=ORANGE; mk.line.fill.background()
def kpi(slide,x,y,num,label):
    box=slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE,Inches(x),Inches(y),Inches(2.25),Inches(.85)); box.fill.solid(); box.fill.fore_color.rgb=RGBColor(248,248,248); box.line.color.rgb=RGBColor(230,230,230)
    tf=box.text_frame; tf.clear(); p=tf.paragraphs[0]; p.alignment=PP_ALIGN.CENTER; r=p.add_run(); r.text=str(num); r.font.size=Pt(22); r.font.bold=True; r.font.color.rgb=NAVY
    p=tf.add_paragraph(); p.alignment=PP_ALIGN.CENTER; r=p.add_run(); r.text=label; r.font.size=Pt(8.5); r.font.color.rgb=GREY
reply=int(base.lead_replied.sum()); sustained=int((base.messages_declared.fillna(0)>=3).sum()); visit=int(base.visit_or_handoff_offered.sum()); tech=int(base.technical_issue_signal.sum())
s=prs.slides.add_slide(prs.slide_layouts[6]); title(s,'Mantra: del lead digital a una conversación que avanza',f'Base: {N:,} PDFs legibles procesados desde Drive')
kpi(s,.6,1.25,f'{N:,}','PDFs legibles'); kpi(s,3.0,1.25,f'{reply/max(N,1):.0%}','lead respondió'); kpi(s,5.4,1.25,f'{sustained/max(N,1):.0%}','conversación >=3 mensajes'); kpi(s,7.8,1.25,f'{visit/max(N,1):.0%}','visita/cita/handoff'); kpi(s,10.2,1.25,f'{tech/max(N,1):.0%}','incidencia técnica')
s.shapes.add_picture(str(FIG/'conversation_funnel.png'),Inches(.75),Inches(2.35),width=Inches(6.4))
tb=s.shapes.add_textbox(Inches(7.55),Inches(2.45),Inches(4.95),Inches(3.5)); tf=tb.text_frame; tf.word_wrap=True
for i,txt in enumerate(['Lectura ejecutiva: medir progresión, no solo volumen de chats.','Pregunta clave: Mantra responde → ¿el lead vuelve a escribir? → ¿se genera visita/handoff?','Decisión: incorporar este bloque después de medios digitales en el reporte comercial.']):
    p=tf.paragraphs[0] if i==0 else tf.add_paragraph(); r=p.add_run(); r.text=txt; r.font.size=Pt(15 if i==0 else 13); r.font.bold=(i==0); r.font.color.rgb=NAVY if i==0 else GREY
s=prs.slides.add_slide(prs.slide_layouts[6]); title(s,'Qué conversaciones explican la oportunidad digital','Cruzar intención, proyecto y profundidad para ajustar guion y handoff')
s.shapes.add_picture(str(FIG/'conversation_projects.png'),Inches(.7),Inches(1.25),width=Inches(5.3)); s.shapes.add_picture(str(FIG/'conversation_depth.png'),Inches(6.4),Inches(1.25),width=Inches(5.3))
box=s.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE,Inches(.75),Inches(5.1),Inches(11.7),Inches(1.25)); box.fill.solid(); box.fill.fore_color.rgb=RGBColor(247,247,247); box.line.color.rgb=RGBColor(225,225,225)
tf=box.text_frame; tf.clear(); r=tf.paragraphs[0].add_run(); r.text='Tres decisiones que habilita'; r.font.size=Pt(15); r.font.bold=True; r.font.color.rgb=NAVY
p=tf.add_paragraph(); r=p.add_run(); r.text='1) Qué temas generan continuidad.  2) En qué proyecto/asesor se rompe la conversación.  3) Qué guion probar para elevar visita/handoff.'; r.font.size=Pt(13); r.font.color.rgb=GREY
ppt=OUT/'conversation_insights_committee_appendix.pptx'; prs.save(ppt); print('PPT generado:', ppt)
